# 1. Analysis - Cash Flow Statement

## Notebook Summary

This notebook analyzes a single company's cash flow statement history from Financial Modeling Prep using the shared ticker in `../../params.py`. It is organized into data loading and preparation, annual and quarterly summary snapshots, core cash flow trend charts, and cash-conversion, working-capital, and capital-allocation diagnostics.

Run the notebook from top to bottom after updating `ticker_str` in `../../params.py` so the helper functions, annual and quarterly datasets, and all downstream tables and charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly cash flow statement history, and calculate the derived cash-generation, working-capital, capital-allocation, and TTM fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Import libraries
from pathlib import Path
import os
import runpy

import pandas as pd
import plotly.graph_objects as go
import requests

In [ ]:
# 2b. Load shared parameters
PARAMS_FILE_CANDIDATES = []
for base_path in [Path.cwd(), *Path.cwd().parents]:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )

for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]
BASE_URL = "https://financialmodelingprep.com/stable"
ANNUAL_LIMIT = 20
QUARTERLY_LIMIT = 40


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
# 2b. Cash flow helper functions
def iter_workspace_roots(start_path: Path | None = None):
    current = (start_path or Path.cwd()).resolve()
    seen = set()
    for candidate in [current, *current.parents]:
        for root in (candidate, candidate / "Investment Scripts"):
            key = str(root)
            if key in seen:
                continue
            seen.add(key)
            yield root


def find_project_root(start_path: Path | None = None) -> Path:
    for candidate in iter_workspace_roots(start_path):
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists() or (candidate / ".env").exists():
            return candidate
    return (start_path or Path.cwd()).resolve()


def load_project_env() -> None:
    for workspace_root in iter_workspace_roots():
        env_path = workspace_root / ".env"
        if not env_path.exists():
            continue

        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if value and value[0] == value[-1] and value[0] in {'"', "'"}:
                value = value[1:-1]
            os.environ.setdefault(key, value)
        return


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def prepare_cash_flow_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if frame.empty:
        return frame

    frame["date"] = pd.to_datetime(frame.get("date"), errors="coerce")

    if "calendarYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["calendarYear"], errors="coerce")
    elif "fiscalYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["fiscalYear"], errors="coerce")
    else:
        frame["calendarYear"] = pd.NA

    if frame["calendarYear"].isna().all():
        frame["calendarYear"] = frame["date"].dt.year

    if "period" not in frame.columns:
        frame["period"] = pd.NA

    numeric_columns = [
        "netIncome",
        "depreciationAndAmortization",
        "deferredIncomeTax",
        "stockBasedCompensation",
        "changeInWorkingCapital",
        "accountsReceivables",
        "inventory",
        "accountsPayables",
        "otherWorkingCapital",
        "operatingCashFlow",
        "netCashProvidedByOperatingActivities",
        "capitalExpenditure",
        "investmentsInPropertyPlantAndEquipment",
        "acquisitionsNet",
        "purchasesOfInvestments",
        "salesMaturitiesOfInvestments",
        "netCashProvidedByInvestingActivities",
        "commonStockRepurchased",
        "commonDividendsPaid",
        "netDividendsPaid",
        "netCashProvidedByFinancingActivities",
        "netChangeInCash",
        "freeCashFlow",
        "cashAtBeginningOfPeriod",
        "cashAtEndOfPeriod",
    ]
    for column in numeric_columns:
        if column not in frame.columns:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame


def apply_standard_figure_layout(fig: go.Figure, title: str, height: int, bottom_margin: int = 140) -> None:
    fig.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_dark",
        paper_bgcolor="#020817",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="x unified",
        hoverlabel={
            "bgcolor": "#0f172a",
            "font": {"color": "#e2e8f0", "size": 13},
            "namelength": -1,
        },
        legend={
            "orientation": "h",
            "yanchor": "top",
            "y": -0.16,
            "x": 0,
            "xanchor": "left",
            "bgcolor": "rgba(2, 8, 23, 0.6)",
        },
        autosize=True,
        height=height,
        margin={"l": 60, "r": 30, "t": 120, "b": bottom_margin},
    )


def format_metric_value(value: float, style: str) -> str:
    if pd.isna(value):
        return "-"
    if style == "currency":
        return f"{value:,.0f}"
    if style == "percent":
        return f"{value:.2%}"
    if style == "ratio":
        return f"{value:,.2f}"
    return f"{value}"

In [ ]:
# 2c. Override workspace discovery for external execution
def iter_workspace_roots(start_path: Path | None = None):
    current = (start_path or Path.cwd()).resolve()
    seen = set()
    for candidate in [current, *current.parents]:
        for root in (candidate, candidate / "Investment Scripts"):
            key = str(root)
            if key in seen:
                continue
            seen.add(key)
            yield root


def find_project_root(start_path: Path | None = None) -> Path:
    for candidate in iter_workspace_roots(start_path):
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists() or (candidate / ".env").exists():
            return candidate
    return (start_path or Path.cwd()).resolve()


def load_project_env() -> None:
    for workspace_root in iter_workspace_roots():
        env_path = workspace_root / ".env"
        if not env_path.exists():
            continue

        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if value and value[0] == value[-1] and value[0] in {'"', "'"}:
                value = value[1:-1]
            os.environ.setdefault(key, value)
        return


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload

In [ ]:
# 3. Fetch annual cash flow statement history
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

annual_cash_flow_statement = prepare_cash_flow_frame(
    pd.DataFrame(request_json("cash-flow-statement", symbol=SYMBOL, limit=ANNUAL_LIMIT))
)

if annual_cash_flow_statement.empty:
    raise RuntimeError(f"FMP did not return annual cash flow history for {SYMBOL}.")

In [ ]:
# 3b. Prepare annual cash flow analytics
annual_cash = (
    annual_cash_flow_statement.loc[:, [
        "date",
        "calendarYear",
        "period",
        "netIncome",
        "depreciationAndAmortization",
        "deferredIncomeTax",
        "stockBasedCompensation",
        "changeInWorkingCapital",
        "accountsReceivables",
        "inventory",
        "accountsPayables",
        "otherWorkingCapital",
        "operatingCashFlow",
        "netCashProvidedByOperatingActivities",
        "capitalExpenditure",
        "investmentsInPropertyPlantAndEquipment",
        "acquisitionsNet",
        "purchasesOfInvestments",
        "salesMaturitiesOfInvestments",
        "netCashProvidedByInvestingActivities",
        "commonStockRepurchased",
        "commonDividendsPaid",
        "netDividendsPaid",
        "netCashProvidedByFinancingActivities",
        "netChangeInCash",
        "freeCashFlow",
        "cashAtBeginningOfPeriod",
        "cashAtEndOfPeriod",
    ]]
    .dropna(subset=["date", "netIncome"])
)
annual_cash["operatingCashFlow"] = annual_cash["operatingCashFlow"].fillna(annual_cash["netCashProvidedByOperatingActivities"])
annual_cash["capitalExpenditure"] = annual_cash["capitalExpenditure"].fillna(annual_cash["investmentsInPropertyPlantAndEquipment"])
annual_cash["netDividendsPaid"] = annual_cash["netDividendsPaid"].fillna(annual_cash["commonDividendsPaid"])
annual_cash["freeCashFlow"] = annual_cash["freeCashFlow"].fillna(annual_cash["operatingCashFlow"] + annual_cash["capitalExpenditure"])
annual_cash["shareholderReturns"] = annual_cash["commonStockRepurchased"].fillna(0) + annual_cash["netDividendsPaid"].fillna(0)
annual_cash["discretionaryCashFlow"] = annual_cash["operatingCashFlow"] + annual_cash["capitalExpenditure"]
annual_cash = annual_cash.sort_values("date").reset_index(drop=True)
annual_cash["operatingCashFlowYoY"] = annual_cash["operatingCashFlow"].pct_change()
annual_cash["freeCashFlowYoY"] = annual_cash["freeCashFlow"].pct_change()
annual_cash["netIncomeYoY"] = annual_cash["netIncome"].pct_change()
annual_cash["netCashChangeYoY"] = annual_cash["netChangeInCash"].pct_change()
annual_cash["cfoToNetIncome"] = annual_cash["operatingCashFlow"].div(annual_cash["netIncome"].replace(0, pd.NA))
annual_cash["capexToCfo"] = annual_cash["capitalExpenditure"].mul(-1).div(annual_cash["operatingCashFlow"].replace(0, pd.NA))
annual_cash["fcfToCfo"] = annual_cash["freeCashFlow"].div(annual_cash["operatingCashFlow"].replace(0, pd.NA))
annual_cash["stockCompToCfo"] = annual_cash["stockBasedCompensation"].div(annual_cash["operatingCashFlow"].replace(0, pd.NA))
annual_cash["workingCapitalDragToCfo"] = annual_cash["changeInWorkingCapital"].mul(-1).div(annual_cash["operatingCashFlow"].replace(0, pd.NA))
annual_cash["buybacksToFcf"] = annual_cash["commonStockRepurchased"].mul(-1).div(annual_cash["freeCashFlow"].replace(0, pd.NA))
annual_cash["dividendsToFcf"] = annual_cash["netDividendsPaid"].mul(-1).div(annual_cash["freeCashFlow"].replace(0, pd.NA))
annual_cash["shareholderReturnsToFcf"] = annual_cash["shareholderReturns"].mul(-1).div(annual_cash["freeCashFlow"].replace(0, pd.NA))

In [ ]:
# 4. Fetch quarterly cash flow statement history
quarterly_cash_flow_statement = prepare_cash_flow_frame(
    pd.DataFrame(request_json("cash-flow-statement", symbol=SYMBOL, period="quarter", limit=QUARTERLY_LIMIT))
)

if quarterly_cash_flow_statement.empty:
    raise RuntimeError(f"FMP did not return quarterly cash flow history for {SYMBOL}.")

In [ ]:
# 4b. Prepare quarterly cash flow analytics
quarterly_cash = (
    quarterly_cash_flow_statement.loc[:, [
        "date",
        "calendarYear",
        "period",
        "netIncome",
        "depreciationAndAmortization",
        "deferredIncomeTax",
        "stockBasedCompensation",
        "changeInWorkingCapital",
        "accountsReceivables",
        "inventory",
        "accountsPayables",
        "otherWorkingCapital",
        "operatingCashFlow",
        "netCashProvidedByOperatingActivities",
        "capitalExpenditure",
        "investmentsInPropertyPlantAndEquipment",
        "acquisitionsNet",
        "purchasesOfInvestments",
        "salesMaturitiesOfInvestments",
        "netCashProvidedByInvestingActivities",
        "commonStockRepurchased",
        "commonDividendsPaid",
        "netDividendsPaid",
        "netCashProvidedByFinancingActivities",
        "netChangeInCash",
        "freeCashFlow",
        "cashAtBeginningOfPeriod",
        "cashAtEndOfPeriod",
    ]]
    .dropna(subset=["date", "netIncome"])
)
quarterly_cash["operatingCashFlow"] = quarterly_cash["operatingCashFlow"].fillna(quarterly_cash["netCashProvidedByOperatingActivities"])
quarterly_cash["capitalExpenditure"] = quarterly_cash["capitalExpenditure"].fillna(quarterly_cash["investmentsInPropertyPlantAndEquipment"])
quarterly_cash["netDividendsPaid"] = quarterly_cash["netDividendsPaid"].fillna(quarterly_cash["commonDividendsPaid"])
quarterly_cash["freeCashFlow"] = quarterly_cash["freeCashFlow"].fillna(quarterly_cash["operatingCashFlow"] + quarterly_cash["capitalExpenditure"])
quarterly_cash["shareholderReturns"] = quarterly_cash["commonStockRepurchased"].fillna(0) + quarterly_cash["netDividendsPaid"].fillna(0)
quarterly_cash["discretionaryCashFlow"] = quarterly_cash["operatingCashFlow"] + quarterly_cash["capitalExpenditure"]
quarterly_cash = quarterly_cash.sort_values("date").reset_index(drop=True)
quarterly_cash["quarterLabel"] = quarterly_cash["period"].astype("string").str.upper().str.extract(r"(Q[1-4])", expand=False)
quarterly_cash.loc[quarterly_cash["quarterLabel"].isna(), "quarterLabel"] = (
    "Q" + quarterly_cash.loc[quarterly_cash["quarterLabel"].isna(), "date"].dt.quarter.astype("Int64").astype(str)
)
quarterly_cash["quarterlyYear"] = quarterly_cash["date"].dt.year
quarterly_cash["operatingCashFlowYoY"] = quarterly_cash["operatingCashFlow"].pct_change(4)
quarterly_cash["freeCashFlowYoY"] = quarterly_cash["freeCashFlow"].pct_change(4)
quarterly_cash["netIncomeYoY"] = quarterly_cash["netIncome"].pct_change(4)
quarterly_cash["netCashChangeYoY"] = quarterly_cash["netChangeInCash"].pct_change(4)
quarterly_cash["cfoToNetIncome"] = quarterly_cash["operatingCashFlow"].div(quarterly_cash["netIncome"].replace(0, pd.NA))
quarterly_cash["capexToCfo"] = quarterly_cash["capitalExpenditure"].mul(-1).div(quarterly_cash["operatingCashFlow"].replace(0, pd.NA))
quarterly_cash["fcfToCfo"] = quarterly_cash["freeCashFlow"].div(quarterly_cash["operatingCashFlow"].replace(0, pd.NA))
quarterly_cash["stockCompToCfo"] = quarterly_cash["stockBasedCompensation"].div(quarterly_cash["operatingCashFlow"].replace(0, pd.NA))
quarterly_cash["workingCapitalDragToCfo"] = quarterly_cash["changeInWorkingCapital"].mul(-1).div(quarterly_cash["operatingCashFlow"].replace(0, pd.NA))
quarterly_cash["buybacksToFcf"] = quarterly_cash["commonStockRepurchased"].mul(-1).div(quarterly_cash["freeCashFlow"].replace(0, pd.NA))
quarterly_cash["dividendsToFcf"] = quarterly_cash["netDividendsPaid"].mul(-1).div(quarterly_cash["freeCashFlow"].replace(0, pd.NA))
quarterly_cash["shareholderReturnsToFcf"] = quarterly_cash["shareholderReturns"].mul(-1).div(quarterly_cash["freeCashFlow"].replace(0, pd.NA))
quarterly_cash["ttmNetIncome"] = quarterly_cash["netIncome"].rolling(4).sum()
quarterly_cash["ttmOperatingCashFlow"] = quarterly_cash["operatingCashFlow"].rolling(4).sum()
quarterly_cash["ttmFreeCashFlow"] = quarterly_cash["freeCashFlow"].rolling(4).sum()
quarterly_cash["ttmCapitalExpenditure"] = quarterly_cash["capitalExpenditure"].rolling(4).sum()
quarterly_cash["ttmShareholderReturns"] = quarterly_cash["shareholderReturns"].rolling(4).sum()
quarterly_cash["ttmNetChangeInCash"] = quarterly_cash["netChangeInCash"].rolling(4).sum()
quarterly_cash["ttmDiscretionaryCashFlow"] = quarterly_cash["discretionaryCashFlow"].rolling(4).sum()
quarterly_cash["ttmBuybacks"] = quarterly_cash["commonStockRepurchased"].rolling(4).sum()
quarterly_cash["ttmDividends"] = quarterly_cash["netDividendsPaid"].rolling(4).sum()
quarterly_cash["ttmCfoToNetIncome"] = quarterly_cash["ttmOperatingCashFlow"].div(quarterly_cash["ttmNetIncome"].replace(0, pd.NA))
quarterly_cash["ttmCapexToCfo"] = quarterly_cash["ttmCapitalExpenditure"].mul(-1).div(quarterly_cash["ttmOperatingCashFlow"].replace(0, pd.NA))
quarterly_cash["ttmFcfToCfo"] = quarterly_cash["ttmFreeCashFlow"].div(quarterly_cash["ttmOperatingCashFlow"].replace(0, pd.NA))
quarterly_cash["ttmShareholderReturnsToFcf"] = quarterly_cash["ttmShareholderReturns"].mul(-1).div(quarterly_cash["ttmFreeCashFlow"].replace(0, pd.NA))
quarterly_cash["ttmBuybacksToFcf"] = quarterly_cash["ttmBuybacks"].mul(-1).div(quarterly_cash["ttmFreeCashFlow"].replace(0, pd.NA))
quarterly_cash["ttmDividendsToFcf"] = quarterly_cash["ttmDividends"].mul(-1).div(quarterly_cash["ttmFreeCashFlow"].replace(0, pd.NA))

In [ ]:
# 5. Build annual summary stats
from IPython.display import display

latest_annual = annual_cash.iloc[-1]
annual_summary = pd.DataFrame(
    [
        {"metric": "Latest annual net income", "value": latest_annual["netIncome"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual operating cash flow", "value": latest_annual["operatingCashFlow"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual capital expenditure", "value": latest_annual["capitalExpenditure"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual free cash flow", "value": latest_annual["freeCashFlow"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual shareholder returns", "value": latest_annual["shareholderReturns"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual CFO to net income", "value": latest_annual["cfoToNetIncome"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual capex to CFO", "value": latest_annual["capexToCfo"], "style": "percent", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual FCF to CFO", "value": latest_annual["fcfToCfo"], "style": "percent", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual shareholder returns to FCF", "value": latest_annual["shareholderReturnsToFcf"], "style": "percent", "asOf": latest_annual["date"].date()},
    ]
)
annual_summary_display = annual_summary.copy()
annual_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(annual_summary_display["value"], annual_summary_display["style"])]
display(annual_summary_display.loc[:, ["metric", "value", "asOf"]])

In [ ]:
# 6. Build quarterly summary stats
from IPython.display import display

latest_quarter = quarterly_cash.iloc[-1]
latest_ttm = quarterly_cash.dropna(subset=["ttmOperatingCashFlow"]).iloc[-1]
quarterly_summary = pd.DataFrame(
    [
        {"metric": "Latest quarterly net income", "value": latest_quarter["netIncome"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly operating cash flow", "value": latest_quarter["operatingCashFlow"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly free cash flow", "value": latest_quarter["freeCashFlow"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM operating cash flow", "value": latest_ttm["ttmOperatingCashFlow"], "style": "currency", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest TTM free cash flow", "value": latest_ttm["ttmFreeCashFlow"], "style": "currency", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest quarterly CFO to net income", "value": latest_quarter["cfoToNetIncome"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM CFO to net income", "value": latest_ttm["ttmCfoToNetIncome"], "style": "ratio", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest quarterly working capital change", "value": latest_quarter["changeInWorkingCapital"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM shareholder returns", "value": latest_ttm["ttmShareholderReturns"], "style": "currency", "asOf": latest_ttm["date"].date()},
    ]
)
quarterly_summary_display = quarterly_summary.copy()
quarterly_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(quarterly_summary_display["value"], quarterly_summary_display["style"])]
display(quarterly_summary_display.loc[:, ["metric", "value", "asOf"]])

## Trend and Cash Generation Charts

These cells move from summary tables into visualization. They cover annual and quarterly cash-generation trends, working-capital and non-cash drivers, investing and financing flows, and the growth profile of net income, operating cash flow, and free cash flow.

In [ ]:
# 7. Plot annual cash flow trends
from plotly.subplots import make_subplots

annual_cash_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.22, 0.24, 0.24],
    subplot_titles=(
        "Annual Net Income, Operating Cash Flow, and Free Cash Flow" ,
        "Annual Non-cash and Working-capital Drivers" ,
        "Annual Investing, Financing, and Net Cash Change" ,
        "Annual Cash Flow Growth Rates" ,
    ),
)

for metric_name, label, color in [
    ("netIncome", "Net income", "#60a5fa"),
    ("operatingCashFlow", "Operating cash flow", "#34d399"),
    ("freeCashFlow", "Free cash flow", "#f59e0b"),
]:
    annual_cash_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
        ),
        row=1,
        col=1,
    )

for metric_name, label, color, dash in [
    ("depreciationAndAmortization", "Depreciation and amortization", "#a78bfa", None),
    ("stockBasedCompensation", "Stock-based compensation", "#fb7185", None),
    ("changeInWorkingCapital", "Change in working capital", "#38bdf8", "dash"),
]:
    line_style = {"color": color, "width": 2.25}
    if dash:
        line_style["dash"] = dash
    annual_cash_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line=line_style,
            marker={"size": 6},
        ),
        row=2,
        col=1,
    )

for metric_name, label, color in [
    ("netCashProvidedByInvestingActivities", "Investing cash flow", "#f97316"),
    ("netCashProvidedByFinancingActivities", "Financing cash flow", "#c084fc"),
    ("netChangeInCash", "Net change in cash", "#22c55e"),
]:
    annual_cash_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
        ),
        row=3,
        col=1,
    )

for metric_name, label, color in [
    ("netIncomeYoY", "Net income growth", "#60a5fa"),
    ("operatingCashFlowYoY", "Operating cash flow growth", "#34d399"),
    ("freeCashFlowYoY", "Free cash flow growth", "#f59e0b"),
]:
    annual_cash_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
        ),
        row=4,
        col=1,
    )

apply_standard_figure_layout(annual_cash_fig, f"{chart_label} annual cash flow statement", 1460)

for row_number in [1, 2, 3, 4]:
    annual_cash_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=1)

annual_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
annual_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=2, col=1)
annual_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=3, col=1)
annual_cash_fig.update_yaxes(title_text="YoY growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
annual_cash_fig.update_xaxes(title_text="Date", row=4, col=1)

annual_cash_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly cash flow trends
from plotly.subplots import make_subplots

quarterly_cash_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.22, 0.24, 0.24],
    subplot_titles=(
        "Quarterly Net Income, Operating Cash Flow, and Free Cash Flow" ,
        "Quarterly Non-cash and Working-capital Drivers" ,
        "Quarterly Investing, Financing, and Net Cash Change" ,
        "Quarterly Cash Flow Growth Rates" ,
    ),
)

for metric_name, label, color in [
    ("netIncome", "Net income", "#60a5fa"),
    ("operatingCashFlow", "Operating cash flow", "#34d399"),
    ("freeCashFlow", "Free cash flow", "#f59e0b"),
]:
    quarterly_cash_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=quarterly_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
        ),
        row=1,
        col=1,
    )

for metric_name, label, color, dash in [
    ("depreciationAndAmortization", "Depreciation and amortization", "#a78bfa", None),
    ("stockBasedCompensation", "Stock-based compensation", "#fb7185", None),
    ("changeInWorkingCapital", "Change in working capital", "#38bdf8", "dash"),
]:
    line_style = {"color": color, "width": 2.0}
    if dash:
        line_style["dash"] = dash
    quarterly_cash_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=quarterly_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line=line_style,
            marker={"size": 6},
        ),
        row=2,
        col=1,
    )

for metric_name, label, color in [
    ("netCashProvidedByInvestingActivities", "Investing cash flow", "#f97316"),
    ("netCashProvidedByFinancingActivities", "Financing cash flow", "#c084fc"),
    ("netChangeInCash", "Net change in cash", "#22c55e"),
]:
    quarterly_cash_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=quarterly_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.0},
            marker={"size": 6},
        ),
        row=3,
        col=1,
    )

for metric_name, label, color in [
    ("netIncomeYoY", "Net income growth", "#60a5fa"),
    ("operatingCashFlowYoY", "Operating cash flow growth", "#34d399"),
    ("freeCashFlowYoY", "Free cash flow growth", "#f59e0b"),
]:
    quarterly_cash_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=quarterly_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.0},
            marker={"size": 6},
        ),
        row=4,
        col=1,
    )

apply_standard_figure_layout(quarterly_cash_fig, f"{chart_label} quarterly cash flow statement", 1460)

for row_number in [1, 2, 3, 4]:
    quarterly_cash_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=1)

quarterly_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
quarterly_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=2, col=1)
quarterly_cash_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=3, col=1)
quarterly_cash_fig.update_yaxes(title_text="YoY growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
quarterly_cash_fig.update_xaxes(title_text="Date", row=4, col=1)

quarterly_cash_fig.show(config={"responsive": True, "displaylogo": False})

## Cash Conversion and Capital Allocation

These cells summarize whether accounting earnings are turning into cash, how much operating cash is being reinvested, and how aggressively management is returning cash through buybacks and dividends relative to free cash flow.

In [ ]:
# 9. Build cash conversion tables
from IPython.display import Markdown, display

annual_conversion_frame = annual_cash.tail(min(len(annual_cash), 6)).copy()
annual_conversion_labels = annual_conversion_frame["calendarYear"].astype("Int64").astype(str)
annual_conversion_table = pd.DataFrame(
    [
        annual_conversion_frame["cfoToNetIncome"].reset_index(drop=True).tolist(),
        annual_conversion_frame["capexToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["fcfToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["stockCompToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["workingCapitalDragToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["buybacksToFcf"].reset_index(drop=True).tolist(),
        annual_conversion_frame["dividendsToFcf"].reset_index(drop=True).tolist(),
        annual_conversion_frame["shareholderReturnsToFcf"].reset_index(drop=True).tolist(),
    ],
    index=[
        "CFO to net income",
        "Capex to CFO",
        "FCF to CFO",
        "Stock comp to CFO",
        "Working-capital drag to CFO",
        "Buybacks to FCF",
        "Dividends to FCF",
        "Shareholder returns to FCF",
    ],
    columns=annual_conversion_labels.tolist(),
)

quarterly_conversion_frame = quarterly_cash.dropna(subset=["ttmOperatingCashFlow"]).tail(min(len(quarterly_cash.dropna(subset=["ttmOperatingCashFlow"])), 8)).copy()
quarterly_conversion_labels = (
    quarterly_conversion_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_conversion_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_conversion_frame["date"].dt.strftime("%b %Y")
    + ")"
)
quarterly_conversion_table = pd.DataFrame(
    [
        quarterly_conversion_frame["ttmCfoToNetIncome"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmCapexToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmFcfToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["stockCompToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["workingCapitalDragToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmBuybacksToFcf"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmDividendsToFcf"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmShareholderReturnsToFcf"].reset_index(drop=True).tolist(),
    ],
    index=[
        "TTM CFO to net income",
        "TTM capex to CFO",
        "TTM FCF to CFO",
        "Quarterly stock comp to CFO",
        "Quarterly working-capital drag to CFO",
        "TTM buybacks to FCF",
        "TTM dividends to FCF",
        "TTM shareholder returns to FCF",
    ],
    columns=quarterly_conversion_labels.tolist(),
)

display(Markdown(f"### {SYMBOL} annual cash conversion and capital allocation"))
display(annual_conversion_table.round(2))
display(Markdown(f"### {SYMBOL} quarterly and TTM cash conversion and capital allocation"))
display(quarterly_conversion_table.round(2))

In [ ]:
# 10. Plot cash conversion and capital allocation
from plotly.subplots import make_subplots

cash_conversion_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    subplot_titles=(
        "Annual cash conversion ratios" ,
        "Quarterly and TTM cash conversion ratios" ,
        "Annual capital allocation as a share of FCF" ,
        "Quarterly and TTM capital allocation as a share of FCF" ,
        "Annual discretionary cash generation" ,
        "TTM discretionary cash generation" ,
    ),
)

for metric_name, label, color in [
    ("cfoToNetIncome", "CFO to net income", "#60a5fa"),
    ("capexToCfo", "Capex to CFO", "#f59e0b"),
    ("fcfToCfo", "FCF to CFO", "#34d399"),
]:
    cash_conversion_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
            legendgroup="conversion" ,
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    cash_conversion_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=quarterly_cash["ttm" + metric_name[0].upper() + metric_name[1:]] if "ttm" + metric_name[0].upper() + metric_name[1:] in quarterly_cash.columns else quarterly_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
            legendgroup="conversion" ,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

for metric_name, label, color in [
    ("buybacksToFcf", "Buybacks to FCF", "#fb7185"),
    ("dividendsToFcf", "Dividends to FCF", "#a78bfa"),
    ("shareholderReturnsToFcf", "Shareholder returns to FCF", "#38bdf8"),
]:
    cash_conversion_fig.add_trace(
        go.Scatter(
            x=annual_cash["date"],
            y=annual_cash[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
            legendgroup="allocation" ,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

for metric_name, source_name, label, color in [
    ("ttmBuybacksToFcf", "buybacksToFcf", "Buybacks to FCF", "#fb7185"),
    ("ttmDividendsToFcf", "dividendsToFcf", "Dividends to FCF", "#a78bfa"),
    ("ttmShareholderReturnsToFcf", "shareholderReturnsToFcf", "Shareholder returns to FCF", "#38bdf8"),
]:
    series = quarterly_cash[metric_name] if metric_name in quarterly_cash.columns else quarterly_cash[source_name]
    cash_conversion_fig.add_trace(
        go.Scatter(
            x=quarterly_cash["date"],
            y=series,
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
            legendgroup="allocation" ,
            showlegend=False,
        ),
        row=2,
        col=2,
    )

for frame, row_number, col_number, discretionary_column, fcf_column, cash_column in [
    (annual_cash, 3, 1, "discretionaryCashFlow", "freeCashFlow", "netChangeInCash"),
    (quarterly_cash.dropna(subset=["ttmDiscretionaryCashFlow"]), 3, 2, "ttmDiscretionaryCashFlow", "ttmFreeCashFlow", "ttmNetChangeInCash"),
]:
    for metric_name, label, color in [
        (discretionary_column, "Discretionary cash flow", "#22c55e"),
        (fcf_column, "Free cash flow", "#f59e0b"),
        (cash_column, "Net change in cash", "#60a5fa"),
    ]:
        cash_conversion_fig.add_trace(
            go.Scatter(
                x=frame["date"],
                y=frame[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.25},
                marker={"size": 7 if col_number == 1 else 6},
                legendgroup="cash-generation" ,
                showlegend=False,
            ),
            row=row_number,
            col=col_number,
        )

apply_standard_figure_layout(cash_conversion_fig, f"{chart_label} cash conversion and capital allocation", 1280)

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        cash_conversion_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=col_number)

cash_conversion_fig.update_yaxes(title_text="Ratio", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=",.2f", row=1, col=1)
cash_conversion_fig.update_yaxes(title_text="Ratio", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=",.2f", row=1, col=2)
cash_conversion_fig.update_yaxes(title_text="Percent of FCF", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=2, col=1)
cash_conversion_fig.update_yaxes(title_text="Percent of FCF", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=2, col=2)
cash_conversion_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=3, col=1)
cash_conversion_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=3, col=2)

cash_conversion_fig.show(config={"responsive": True, "displaylogo": False})